<a href="https://colab.research.google.com/github/e-saidha/skyhack_querykings/blob/main/Deliverables_1_Exploratory_Data_Analysis_(EDA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

flights = pd.read_csv('/content/Flight Level Data.csv')
pnr_flight = pd.read_csv('/content/PNR+Flight+Level+Data.csv')
pnr_rem = pd.read_csv('/content/PNR Remark Level Data.csv')
bag_level = pd.read_csv('/content/Bag+Level+Data.csv')
airports = pd.read_csv('/content/Airports Data.csv')
airports = airports.drop_duplicates()

In [ ]:
import duckdb, numpy as np
con = duckdb.connect()


con.register("flights", flights)
con.register("pnr_flight", pnr_flight)
con.register("pnr_rem", pnr_rem)
con.register("bag_level", bag_level)
con.register("airports", airports)


In [ ]:

con.execute("""
CREATE OR REPLACE VIEW v_flights_ord AS
SELECT *
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY company_id, flight_number, scheduled_departure_date_local,
                   scheduled_departure_station_code, scheduled_arrival_station_code
      ORDER BY COALESCE(actual_departure_datetime_local, scheduled_departure_datetime_local) DESC
    ) AS rn
  FROM flights
  WHERE scheduled_departure_station_code = 'ORD'
) t
WHERE rn = 1
""")
n_flights = con.execute("SELECT COUNT(*) FROM v_flights_ord").fetchone()[0]



In [ ]:
# 2a) Collapse to one row per (flight+record_locator): take the latest PNR snapshot
con.execute("""
CREATE OR REPLACE VIEW v_pnr_per_booking AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  record_locator,
  -- latest snapshot per booking (handle updates)
  MAX(COALESCE(total_pax,0))                                              AS pnr_pax,
  MAX(COALESCE(lap_child_count,0))                                        AS pnr_lap_childs,
  MAX(COALESCE(CAST(basic_economy_ind AS BIGINT),0))                      AS pnr_basic_econ,
  MAX(
    CASE
      WHEN UPPER(TRIM(CAST(is_stroller_user AS VARCHAR))) IN ('Y','YES','TRUE','1') THEN 1
      ELSE 0
    END
  )                                                                        AS pnr_stroller_user
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY company_id, flight_number, scheduled_departure_date_local,
                   scheduled_departure_station_code, scheduled_arrival_station_code,
                   record_locator
      ORDER BY pnr_creation_date DESC
    ) AS rn
  FROM pnr_flight
) q
WHERE rn = 1
GROUP BY 1,2,3,4,5,6
""")

# 2b) Aggregate per flight (sum across distinct PNRs)
con.execute("""
CREATE OR REPLACE VIEW v_pnr_agg AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  SUM(pnr_pax)           AS pax_total,
  SUM(pnr_lap_childs)    AS lap_childs,
  SUM(pnr_basic_econ)    AS basic_econ,
  SUM(pnr_stroller_user) AS stroller_users,
  COUNT(*)               AS distinct_pnrs
FROM v_pnr_per_booking
GROUP BY 1,2,3,4,5
""")


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW v_bags_agg AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  -- origin == checked
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(bag_type))='ORIGIN' THEN bag_tag_unique_number END)        AS checked_bags,
  -- transfer includes both 'transfer' and 'hot transfer'
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(bag_type)) IN ('TRANSFER','HOT TRANSFER')
                      THEN bag_tag_unique_number END)                                           AS transfer_bags
FROM bag_level
GROUP BY 1,2,3,4,5
""")


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW v_ssr_agg AS
SELECT
  p.company_id, p.flight_number, p.scheduled_departure_date_local,
  p.scheduled_departure_station_code, p.scheduled_arrival_station_code,
  COUNT(*) AS ssr_count
FROM pnr_rem r
JOIN v_pnr_per_booking p
  ON r.record_locator = p.record_locator
 AND r.flight_number  = p.flight_number
GROUP BY 1,2,3,4,5
""")


In [ ]:
df_master = con.execute("""
WITH base AS (
  SELECT
    f.*,
    (f.scheduled_ground_time_minutes - f.minimum_turn_minutes) AS slack_mins,
    EXTRACT(hour FROM CAST(f.scheduled_departure_datetime_local AS TIMESTAMP)) AS dep_hour,
    EXTRACT(dow  FROM CAST(f.scheduled_departure_datetime_local AS TIMESTAMP)) AS dep_dow
  FROM v_flights_ord f
)
SELECT
  b.*,
  COALESCE(p.pax_total,0)        AS pax_total,
  COALESCE(p.lap_childs,0)       AS lap_childs,
  COALESCE(p.basic_econ,0)       AS basic_econ,
  COALESCE(p.stroller_users,0)   AS stroller_users,
  COALESCE(p.distinct_pnrs,0)    AS distinct_pnrs,
  COALESCE(bg.checked_bags,0)    AS checked_bags,
  COALESCE(bg.transfer_bags,0)   AS transfer_bags,
  CASE WHEN COALESCE(bg.checked_bags,0) > 0
       THEN CAST(bg.transfer_bags AS DOUBLE)/bg.checked_bags
       ELSE NULL END             AS transfer_ratio,
  COALESCE(s.ssr_count,0)        AS ssr_count,
  apt.iso_country_code           AS arrival_country
FROM base b
LEFT JOIN v_pnr_agg  p  USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN v_bags_agg bg USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN v_ssr_agg  s  USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN airports apt
       ON b.scheduled_arrival_station_code = apt.airport_iata_code
""").df()

print("Master rows:", len(df_master))
assert len(df_master) == n_flights, "Row-count mismatch: master != flights_ORD"


In [ ]:
# timestamp → delay (mins)
df_master["scheduled_departure_datetime_local"] = pd.to_datetime(df_master["scheduled_departure_datetime_local"])
df_master["actual_departure_datetime_local"]    = pd.to_datetime(df_master["actual_departure_datetime_local"])
df_master["dep_delay_min"] = (
    df_master["actual_departure_datetime_local"] - df_master["scheduled_departure_datetime_local"]
).dt.total_seconds() / 60

# load factor (should not be crazy)
df_master["load_factor"] = df_master["pax_total"] / df_master["total_seats"]

# Hard validations:
n_overcap = int((df_master["pax_total"] > df_master["total_seats"]).sum())
print("Flights with pax_total > total_seats:", n_overcap)

# If any, show the worst 5 for inspection (they may be true anomalies; should be very few)
if n_overcap > 0:
    display(df_master.loc[df_master["pax_total"] > df_master["total_seats"],
                          ["company_id","flight_number","scheduled_departure_date_local",
                           "pax_total","total_seats"]].sort_values("pax_total", ascending=False).head())

print(df_master["load_factor"].describe())
print("Sum checked bags:", int(df_master["checked_bags"].sum()))
print("Sum transfer bags:", int(df_master["transfer_bags"].sum()))
print("Sum ssr_count:", int(df_master["ssr_count"].sum()))


EDA 1

In [ ]:
import pandas as pd


df_master["scheduled_departure_datetime_local"] = pd.to_datetime(df_master["scheduled_departure_datetime_local"])
df_master["actual_departure_datetime_local"] = pd.to_datetime(df_master["actual_departure_datetime_local"])


df_master["dep_delay_min"] = (
    (df_master["actual_departure_datetime_local"] - df_master["scheduled_departure_datetime_local"])
    .dt.total_seconds() / 60
)


In [ ]:
avg_delay = df_master["dep_delay_min"].mean()
late_flights_pct = (df_master["dep_delay_min"] > 15).mean() * 100

print(f"Average Departure Delay: {avg_delay:.2f} minutes")
print(f"Percentage of Flights Delayed >15 min: {late_flights_pct:.2f}%")


In [ ]:
!apt-get install -y fonts-freefont-ttf


In [ ]:

plt.figure(figsize=(9,6))
sns.ecdfplot(df_master["dep_delay_min"], color="navy")

# 15-min cutoff
plt.axvline(15, color="orange", linestyle="--", label="15 min threshold")
plt.axhline(late_flights_pct/100, color="red", linestyle="--", alpha=0.7)
plt.text(15+1, late_flights_pct/100, f"{late_flights_pct:.1f}%",
         color="red", fontsize=11, va="bottom", weight="bold")

plt.title("ECDF of Departure Delays", fontsize=14, weight="bold")
plt.xlabel("Departure Delay (minutes)")
plt.ylabel("Cumulative Proportion of Flights")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


EDA 2

In [ ]:
df_master["slack_mins"] = df_master["scheduled_ground_time_minutes"] - df_master["minimum_turn_minutes"]


In [ ]:
critical = (df_master["slack_mins"] <= 0).sum()
tight = ((df_master["slack_mins"] > 0) & (df_master["slack_mins"] <= 10)).sum()
total = len(df_master)

print(f"Flights at/below minimum turn: {critical} ({critical/total*100:.1f}%)")
print(f"Flights within 10 min slack: {tight} ({tight/total*100:.1f}%)")


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


df_master["slack_mins"] = (
    df_master["scheduled_ground_time_minutes"] - df_master["minimum_turn_minutes"]
)


total = len(df_master)
critical = (df_master["slack_mins"] <= 0).sum()
tight = ((df_master["slack_mins"] > 0) & (df_master["slack_mins"] <= 10)).sum()
comfortable = total - critical - tight

def pct(x):
    return 100 * x / total


print("Flights at/below minimum (≤0):", critical, f"({pct(critical):.1f}%)")
print("Flights with tight slack (1–10):", tight, f"({pct(tight):.1f}%)")
print("Flights comfortable (>10):", comfortable, f"({pct(comfortable):.1f}%)")
print("Total flights:", total)


fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

categories = ["Critical\n(≤0 min)", "Tight\n(1-10 min)", "Comfortable\n(>10 min)"]
counts = [critical, tight, comfortable]
percentages = [pct(critical), pct(tight), pct(comfortable)]


colors = ["#dc2626", "#f59e0b", "#10b981"]
y_positions = np.arange(len(categories))


bars = ax.barh(y_positions, counts, height=0.6, color=colors,
               edgecolor="white", linewidth=2, alpha=0.85)


for i, (bar, color) in enumerate(zip(bars, colors)):
    width = bar.get_width()

    ax.barh(i, width * 0.3, height=0.6, left=0,
            color=color, alpha=0.3, edgecolor="none")


for i, pct_val in enumerate(percentages):
    ax.text(counts[i] / 2, i, f"{pct_val:.1f}%",
            va="center", ha="center", fontsize=14, weight="bold",
            color="white", alpha=0.95)


ax.set_yticks(y_positions)
ax.set_yticklabels(categories, fontsize=12, weight="600")
ax.set_xlabel("Number of Flights", fontsize=12, weight="600", color="#374151")
ax.set_title("Ground Time Slack Distribution",
             fontsize=16, weight="bold", pad=20, color="#1f2937",
             loc="left")


ax.text(0, len(categories) + 0.3,
        "Scheduled ground time minus minimum turnaround requirements",
        fontsize=10, color="#6b7280", style="italic")


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#d1d5db")
ax.spines["bottom"].set_color("#d1d5db")


ax.grid(axis="x", linestyle="--", alpha=0.2, linewidth=0.8)
ax.set_axisbelow(True)


ax.set_facecolor("#f9fafb")
fig.patch.set_facecolor("white")

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

plt.tight_layout()
plt.show()

EDA 3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


valid = df_master[df_master["checked_bags"] > 0].copy()


valid["transfer_ratio"] = valid["transfer_bags"] / valid["checked_bags"]
valid["transfer_share"] = valid["transfer_bags"] / (valid["transfer_bags"] + valid["checked_bags"])


unweighted_mean_ratio = valid["transfer_ratio"].mean()
weighted_mean_ratio   = valid["transfer_bags"].sum() / valid["checked_bags"].sum()

unweighted_mean_share = valid["transfer_share"].mean()
weighted_mean_share   = valid["transfer_bags"].sum() / (valid["transfer_bags"].sum() + valid["checked_bags"].sum())

print(f"Unweighted mean ratio (transfer/checked): {unweighted_mean_ratio:.3f}")
print(f"Weighted mean ratio   (transfer/checked): {weighted_mean_ratio:.3f}")
print(f"Unweighted mean share (transfer/total):   {unweighted_mean_share:.3f}")
print(f"Weighted mean share   (transfer/total):   {weighted_mean_share:.3f}")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np


by_dest = (
    valid.groupby("scheduled_arrival_station_code")[["transfer_bags","checked_bags"]]
         .sum()
         .query("checked_bags > 0")
         .assign(weighted_ratio=lambda d: d["transfer_bags"]/d["checked_bags"],
                 transfer_share=lambda d: d["transfer_bags"]/(d["transfer_bags"]+d["checked_bags"]))
)

top15 = by_dest.sort_values("transfer_share", ascending=False).head(15)


fig, ax = plt.subplots(figsize=(11, 7), dpi=150)

y_positions = np.arange(len(top15))
x_values = top15["transfer_share"].values
destinations = top15.index

colors = plt.cm.Blues(np.linspace(0.7, 0.4, len(top15)))


bars = ax.barh(y_positions, x_values, height=0.65,
               color=colors, edgecolor="white", linewidth=2, alpha=0.9)

for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.barh(i, width * 0.4, height=0.65, left=0,
            color=colors[i], alpha=0.3, edgecolor="none")


for i, (dest, val) in enumerate(zip(destinations, x_values)):

    if val > 0.15:
        ax.text(val - 0.01, i, f"{val:.1%}",
                va="center", ha="right", fontsize=11, weight="bold",
                color="white")
    else:
        ax.text(val + 0.01, i, f"{val:.1%}",
                va="center", ha="left", fontsize=11, weight="bold",
                color="#1f2937")


ax.set_yticks(y_positions)
ax.set_yticklabels(destinations, fontsize=11, weight="600", color="#374151")
ax.set_xlabel("Transfer Share (Transfer Bags / Total Bags)",
              fontsize=12, weight="600", color="#374151")


ax.set_title("Top 15 Destinations by Transfer Bag Share",
             fontsize=16, weight="bold", pad=15, color="#1f2937", loc="left")


ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.0%}'))
ax.set_xlim(0, max(x_values) * 1.12)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#d1d5db")
ax.spines["bottom"].set_color("#d1d5db")


ax.grid(axis="x", linestyle="--", alpha=0.2, linewidth=0.8)
ax.set_axisbelow(True)


ax.set_facecolor("#f9fafb")
fig.patch.set_facecolor("white")


if max(x_values) > 0.5:
    ax.axvline(0.5, color="#94a3b8", linestyle=":", linewidth=1.5, alpha=0.5)
    ax.text(0.5, -0.7, "50%", ha="center", fontsize=9, color="#94a3b8")

plt.tight_layout()
plt.show()

EDA 4

In [ ]:
import pandas as pd

# Create bins
df_master["load_bin"] = pd.cut(
    df_master["load_factor"],
    bins=[0, 0.6, 0.8, 0.9, 1.0, 1.1],
    labels=["0–0.6","0.6–0.8","0.8–0.9","0.9–1.0",">1.0"]
)


In [ ]:
heatmap_data = df_master.groupby(["load_bin","dep_dow"])["dep_delay_min"].mean().unstack()

# Order days properly (Mon–Sun, assuming dep_dow = 0=Mon)
dow_order = [0,1,2,3,4,5,6]
heatmap_data = heatmap_data[dow_order]
heatmap_data.columns = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


sns.set_style("whitegrid")



fig, ax = plt.subplots(figsize=(10, 6), dpi=150)


sns.histplot(df_master["load_factor"], bins=30, kde=True,
             color="#0369a1", alpha=0.75, edgecolor="white",
             linewidth=0.8, line_kws={"linewidth": 3, "color": "#0284c7"},
             ax=ax)


mean_lf = df_master["load_factor"].mean()
median_lf = df_master["load_factor"].median()


ax.axvline(mean_lf, color="#dc2626", linestyle="--", linewidth=2.5,
           alpha=0.3, zorder=10)
ax.text(mean_lf, ax.get_ylim()[1] * 0.95, f"Mean\n{mean_lf:.2%}",
        color="#dc2626", fontsize=11, weight="bold", ha="center",
        va="top", bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                           edgecolor="#dc2626", alpha=0.95, linewidth=1.5))


ax.axvline(median_lf, color="#f59e0b", linestyle="--", linewidth=2.5,
           alpha=0.3, zorder=10)
ax.text(median_lf, ax.get_ylim()[1] * 0.82, f"Median\n{median_lf:.2%}",
        color="#f59e0b", fontsize=11, weight="bold", ha="center",
        va="top", bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                           edgecolor="#f59e0b", alpha=0.95, linewidth=1.5))


ax.set_title("Passenger Load Factor Distribution",
             fontsize=16, weight="bold", pad=20, color="#1f2937")
ax.text(0.5, 1.03, "Capped at 100% capacity",
        transform=ax.transAxes, fontsize=10, color="#6b7280",
        style="italic", ha="center")

ax.set_xlabel("Load Factor", fontsize=12, weight="600", color="#374151")
ax.set_ylabel("Number of Flights", fontsize=12, weight="600", color="#374151")


ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

ax.grid(True, alpha=0.2, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_facecolor("#f9fafb")
fig.patch.set_facecolor("white")

for spine in ax.spines.values():
    spine.set_color("#d1d5db")
    spine.set_linewidth(1)

plt.tight_layout()
plt.show()


df_master["load_bin"] = pd.cut(
    df_master["load_factor"],
    bins=[0, 0.6, 0.8, 0.9, 1.0],
    labels=["0–60%", "60–80%", "80–90%", "90–100%"]
)

delay_by_load = df_master.groupby("load_bin", observed=True)["dep_delay_min"].mean()

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)


colors = ["#10b981", "#f59e0b", "#f97316", "#dc2626"]
x_pos = np.arange(len(delay_by_load))


bars = ax.bar(x_pos, delay_by_load.values, width=0.65,
              color=colors, edgecolor="white", linewidth=2, alpha=0.85)


for i, (bar, color) in enumerate(zip(bars, colors)):
    height = bar.get_height()
    ax.bar(i, height * 0.35, width=0.65, bottom=0,
           color=color, alpha=0.3, edgecolor="none")


for i, val in enumerate(delay_by_load.values):
    ax.text(i, val + max(delay_by_load.values) * 0.02, f"{val:.1f} min",
            ha="center", va="bottom", fontsize=12, weight="bold",
            color="#1f2937")


ax.set_xticks(x_pos)
ax.set_xticklabels(delay_by_load.index, fontsize=11, weight="600",
                   color="#374151")
ax.set_xlabel("Load Factor Range", fontsize=12, weight="600",
              color="#374151")
ax.set_ylabel("Average Departure Delay (minutes)", fontsize=12,
              weight="600", color="#374151")

ax.set_title("Impact of Load Factor on Departure Delays",
             fontsize=16, weight="bold", pad=20, color="#1f2937", loc="left")



ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#d1d5db")
ax.spines["bottom"].set_color("#d1d5db")


ax.grid(axis="y", linestyle="--", alpha=0.2, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_facecolor("#f9fafb")
fig.patch.set_facecolor("white")


ax.axhline(0, color="#94a3b8", linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()


EDA 5


In [ ]:
df_master["ssr_per_100pax"] = df_master["ssr_count"] / (df_master["pax_total"]+1) * 100



In [ ]:
import numpy as np
import pandas as pd


df_master["load_bin"] = pd.cut(
    df_master["load_factor"],
    bins=[0, 0.7, 0.85, 1.0],
    labels=["Low Load", "Medium Load", "High Load"]
)


df_master["ssr_bin"] = pd.qcut(
    df_master["ssr_per_100pax"],
    q=3,
    labels=["Low SSR", "Medium SSR", "High SSR"],
    duplicates="drop"
)



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

pivot = df_master.pivot_table(index="load_bin", columns="ssr_bin", values="dep_delay_min", aggfunc="mean")

plt.figure(figsize=(10,5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Reds", cbar_kws={'label': 'Avg Delay (min)'})
plt.title("Avg Departure Delay by Load & SSR Intensity", fontsize=14, weight="bold")
plt.xlabel("SSR Intensity"); plt.ylabel("Load Factor Bucket")
plt.show()


EDA 6

In [ ]:
avg_delay_by_hour = df_master.groupby("dep_hour")["dep_delay_min"].mean().reset_index()

plt.figure(figsize=(10,5))
sns.lineplot(x="dep_hour", y="dep_delay_min", data=avg_delay_by_hour, marker="o", color="navy")
plt.axhline(15, color="orange", linestyle="--", label="15 min cutoff")

plt.title("Average Departure Delay by Hour of Day", fontsize=14, weight="bold")
plt.xlabel("Departure Hour"); plt.ylabel("Avg Delay (minutes)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


EDA 7

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


avg_delay_by_dest = (
    df_master.groupby("scheduled_arrival_station_code")["dep_delay_min"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)


y_positions = np.arange(len(avg_delay_by_dest))
x_values = avg_delay_by_dest.values
destinations = avg_delay_by_dest.index


bars = ax.barh(y_positions, x_values, height=0.6,
               color="#2563eb", edgecolor="none", alpha=0.85)


for i, val in enumerate(x_values):
    ax.text(val + 0.5, i, f"{val:.1f}",
            va="center", ha="left", fontsize=11,
            color="#1f2937", weight="600")


ax.set_yticks(y_positions)
ax.set_yticklabels(destinations, fontsize=11, color="#374151")
ax.set_xlabel("Average Departure Delay (minutes)",
              fontsize=11, color="#374151")

ax.set_title("Top 10 Destinations by Average Departure Delay",
             fontsize=14, weight="600", pad=15, color="#111827")


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#e5e7eb")
ax.spines["bottom"].set_linewidth(1)


ax.grid(axis="x", linestyle="-", alpha=0.1, linewidth=1, color="#9ca3af")
ax.set_axisbelow(True)


ax.set_facecolor("white")
fig.patch.set_facecolor("white")


ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', colors="#6b7280")

plt.tight_layout()
plt.show()